# Week 5-0 — 전처리: 참고문헌 노이즈 제거 (Reference Section Removal)

**배경**: 로우레벨 검증(`week5_0_lowlevel_check`)에서 한글 문서 뒤쪽 **참고문헌·색인이 노이즈 chunk로 다수 인덱싱**됨을 발견했다. 검색 기법(Hybrid/Rerank)을 얹기 전에 인덱스 밑단부터 정리한다.

**방식**: 헤딩 기준(A안) — 문서별로 "참고문헌/References" 시작 페이지를 찾아 그 이후를 제외한다.

**4주차 교훈 적용**: 클렌징이 검색 앵커까지 지워 실패했던 전철을 밟지 않도록, (1) 지우기 전 위치를 정확히 스캔하고 (2) 잘린 경계를 **육안 확인**한 뒤 (3) 제거 전/후를 **실측 비교**해서 채택 여부를 결정한다.

**판정 기준**: 제거 후 한국어 문항 Context Precision이 **개선되면 채택**, 악화되면 기각(원본 유지).

---
## 1. 설정

In [1]:
from pathlib import Path
import os, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"

load_dotenv(PROJECT_ROOT / ".env"); load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 없음"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
GEN_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o-mini"   # 5주차부터 judge 통일
TOP_K = 5

# 청킹: 4주차 확정 G2 고정
CHUNK_SIZE_BY_LANG    = {"ko": 540, "en": 620, "unknown": 580}
CHUNK_OVERLAP_BY_LANG = {"ko": 80,  "en": 90,  "unknown": 85}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GEN/JUDGE:", GEN_MODEL, "/", JUDGE_MODEL)

PROJECT_ROOT: /Users/jian/Documents/rag-agent-portfolio
GEN/JUDGE: gpt-4o-mini / gpt-4o-mini


---
## 2. 참고문헌 시작 위치 스캔 (API 0원)

각 PDF에서 "참고문헌 / References" 류 헤딩이 등장하는 페이지를 찾는다. **자동 감지는 후보 제시용**이고, 최종 컷 페이지는 다음 셀에서 사람이 확정한다.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
import re

REF_PATTERNS = [
    r"^\s*참\s*고\s*문\s*헌",
    r"^\s*References?\s*$",
    r"^\s*REFERENCES?\s*$",
    r"^\s*Bibliography\s*$",
    r"^\s*색\s*인\s*$",
    r"^\s*Index\s*$",
]

def scan_ref_pages(pdf_path):
    """헤딩 후보가 나타나는 (페이지번호, 매칭줄) 목록"""
    hits = []
    for d in PyMuPDFLoader(str(pdf_path)).load():
        page = d.metadata.get("page", -1)
        for line in d.page_content.split("\n")[:15]:  # 페이지 상단 15줄만 (헤딩은 위쪽)
            for pat in REF_PATTERNS:
                if re.match(pat, line.strip()):
                    hits.append((page, line.strip()[:40]))
    return hits

pdf_files = sorted((DATA_RAW / "pdf").rglob("*.pdf"))
total_pages = {}
for pdf in pdf_files:
    docs = PyMuPDFLoader(str(pdf)).load()
    total_pages[pdf.name] = len(docs)
    hits = scan_ref_pages(pdf)
    print(f"\n=== {pdf.name} (총 {len(docs)}p) ===")
    if hits:
        for pg, line in hits[:10]:
            print(f"  p.{pg}: {line}")
    else:
        print("  (헤딩 후보 없음)")


=== esmo_breast_cancer_patient_guide_korean.pdf (총 70p) ===
  p.60: 참고문헌

=== kbcs_korean_breast_cancer_guideline_2023.pdf (총 270p) ===
  p.41: 참고문헌
  p.44: 참고문헌
  p.45: 참고문헌
  p.47: 참고문헌
  p.66: 참고문헌
  p.69: 참고문헌
  p.79: 참고문헌
  p.89: 참고문헌
  p.95: 참고문헌
  p.99: 참고문헌

=== ncc_breast_cancer_hormone_therapy.pdf (총 2p) ===
  (헤딩 후보 없음)

=== ncc_breast_cancer_prevention_guide.pdf (총 10p) ===
  (헤딩 후보 없음)

=== ncc_breast_cancer_screening_guideline_2015.pdf (총 114p) ===
  p.109: 참고문헌
  p.110: 참고문헌
  p.112: 참고문헌

=== nccn_breast_cancer_screening_diagnosis_patient.pdf (총 52p) ===
  p.49: Index
  p.49: Index

=== nccn_dcis_patient.pdf (총 52p) ===
  p.49: Index
  p.49: Index

=== nccn_inflammatory_breast_cancer_patient.pdf (총 74p) ===
  p.71: Index
  p.71: Index

=== nccn_invasive_breast_cancer_patient.pdf (총 86p) ===
  p.83: Index
  p.83: Index

=== nccn_metastatic_breast_cancer_patient.pdf (총 66p) ===
  p.63: Index
  p.63: Index


### 2-1. kbcs 예외 처리 — 페이지 인용-밀도 필터

**문제**: 헤딩 스캔 결과, kbcs 문서는 참고문헌이 문서 끝에 한 덩어리로 있지 않고 **챕터마다 산재**(p.41, 44, 47, 66, ...)해 있다. 따라서 "p.X부터 끝까지 컷" 방식이 통하지 않는다. (원래 계획했던 방식)

**대응**: 헤딩 위치 대신 **페이지 단위 인용-밀도**로 판정한다 — 페이지 줄의 50% 이상이 인용 패턴(연도;권(호):페이지, et al, 저널명 등)이면 참고문헌 페이지로 간주하고 개별 제외한다.

**검증**: 자동 판정을 그대로 믿지 않고(4주차 클렌징 교훈), ① 걸린 페이지 샘플이 진짜 인용 목록인지, ② 걸린 구간 사이의 '안 걸린' 페이지(갭)가 본문인지 한글식 인용인지 육안으로 확인한 뒤 최종 제외 목록을 확정한다.

In [6]:
# ==== kbcs용: 페이지 인용-밀도 스캔 (챕터별 산재 참고문헌 대응) ====
CITE_PAT = re.compile(r"\d{4};\d+|\bet al\b|J Clin Oncol|N Engl J Med|Cancer\.|\d+\(\d+\):\d+")

def citation_density(text):
    lines = [l for l in text.split("\n") if l.strip()]
    if not lines: return 0.0
    hits = sum(1 for l in lines if CITE_PAT.search(l))
    return hits / len(lines)

target = next(p for p in pdf_files if p.name == "kbcs_korean_breast_cancer_guideline_2023.pdf")
ref_pages = []
for d in PyMuPDFLoader(str(target)).load():
    dens = citation_density(d.page_content)
    if dens >= 0.35:  # 줄의 50% 이상이 인용 패턴이면 참고문헌 페이지로 간주
    	ref_pages.append((d.metadata.get("page"), round(dens, 2)))

print(f"kbcs 인용-밀도 50%+ 페이지: {len(ref_pages)}개")
print([p for p, _ in ref_pages])

kbcs 인용-밀도 50%+ 페이지: 82개
[50, 51, 53, 56, 100, 101, 103, 104, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 163, 164, 167, 168, 169, 170, 171, 172, 173, 174, 221, 222, 223, 224, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252]


In [4]:
# ==== 육안 확인: 걸린 페이지 샘플 + 사이에 빠진 페이지 ====
docs_kbcs = {d.metadata.get("page"): d for d in PyMuPDFLoader(str(target)).load()}
flagged = [p for p, _ in ref_pages]

# 걸린 페이지 중 3개 샘플 (구간별 하나씩)
check_flagged = [109, 172, 240]
# 걸린 구간 사이에 '안 걸린' 페이지들 (수상한 갭)
check_gaps = [110, 113, 120, 128, 239]

for tag, pages in [("걸림(제외 예정)", check_flagged), ("갭(안 걸림 — 본문? 한글인용?)", check_gaps)]:
    print("=" * 70)
    print(f"### {tag}")
    for pg in pages:
        if pg not in docs_kbcs: continue
        print(f"\n--- p.{pg} (앞 350자)")
        print(docs_kbcs[pg].page_content.strip()[:350])
    print()

### 걸림(제외 예정)

--- p.109 (앞 350자)
110 | 
2023 제10차 한국유방암 진료권고안
110 | 
2023 제10차 한국유방암 진료권고안
107.	 Hughes KS, Schnaper LA, Berry D, Cirrincione C, McCormick B, Shank B, et al. Lumpectomy 
plus tamoxifen with or without irradiation in women 70 years of age or older with early breast 
cancer. N Engl J Med. 2004;351(10):971-7.
108.	 Fyles AW, McCready DR, Manchul LA, Trudeau ME, Me

--- p.172 (앞 350자)
2023 The 10
th Korean Clinical Practice Guideline for Breast Cancer | 173
제2장 조기 유방암
제3장 재발 및 전이성 유방암
제4장 유전성 유방암 
제1장 비침습 유방암
markers. The International journal of biological markers. 2007;22(1):24-33.
105.	 Hortobagyi GN, Van Poznak C, Harker WG, Gradishar WJ, Chew H, Dakhil SR, et al. Continued 
treatment effect of zoledronic acid dosing every

--- p.240 (앞 350자)
2023 The 10
th Korean Clinical Practice Guideline for Breast Cancer | 241
제2장 조기 유방암
제3장 재발 및 전이성 유방암
제4장 유전성 유방암 
제1장 비침습 유방암
Surg 2009;96:1-2.
223.	 Finch A, Beiner M, Lubinski J, Lynch HT, Moller P, Rosen B, et al. Salp

In [7]:
new_flagged = [p for p, _ in ref_pages if p not in [109, 111, 115, 117, 121, 123, 124, 125, 126, 127, 129, 130, 131, 132, 135, 137, 139, 141, 172, 173, 174, 223, 232, 233, 234, 235, 236, 237, 238, 240, 241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252]]
print(f"새로 걸린 페이지 {len(new_flagged)}개: {new_flagged}")
for pg in new_flagged[:6]:  # 앞 6개만 육안
    d = docs_kbcs.get(pg)
    print(f"\n--- p.{pg} (밀도 {citation_density(d.page_content):.2f}, 앞 250자)")
    print(d.page_content.strip()[:250])

새로 걸린 페이지 41개: [50, 51, 53, 56, 100, 101, 103, 104, 108, 110, 112, 113, 114, 116, 118, 119, 120, 122, 128, 133, 134, 136, 138, 140, 163, 164, 167, 168, 169, 170, 171, 221, 222, 224, 227, 228, 229, 230, 231, 239, 246]

--- p.50 (밀도 0.37, 앞 250자)
2023 The 10
th Korean Clinical Practice Guideline for Breast Cancer | 51
제2장 조기 유방암
제3장 재발 및 전이성 유방암
제4장 유전성 유방암 
제1장 비침습 유방암
38.	
Corral CJ, Mustoe TA. Controversy in breast reconstruction. Surg Clin North Am 1996;76:309-
26.
39.	
Vezeridis MP, 

--- p.51 (밀도 0.40, 앞 250자)
52 | 
2023 제10차 한국유방암 진료권고안
52 | 
2023 제10차 한국유방암 진료권고안
clinical practice guideline update. Journal of Clinical Oncology 2014;32:1365-83.
50.	
Cody HS, 3rd. Sentinel lymph node biopsy for DCIS: are we approaching consensus? Ann Surg 
Oncol 2007;14:

--- p.53 (밀도 0.38, 앞 250자)
54 | 
2023 제10차 한국유방암 진료권고안
54 | 
2023 제10차 한국유방암 진료권고안
in situ (DCIS). Breast cancer research and treatment 2014;143:343-50.
69.	
Kim K, Jung S-Y, Shin KH, Kim JH, Han W, Lee H-B, et al. Recurre

### 컷 페이지 확정 

자동 스캔은 후보 제시용이며, 최종 컷 페이지는 경계 육안 확인 후 직접 확정.
- 참고문헌이 없는 문서는 목록에서 제외 → 전체 유지.
- kbcs는 참고문헌이 챕터별로 산재해 별도 처리(2-1 인용-밀도 필터).
- 주의: PyMuPDF의 page 메타는 **0-base** (p.269 = 문서의 270쪽).

In [8]:
# ==== 사람이 확정하는 컷 페이지 (스캔 결과 보고 채워짐) ====
KBCS_REF_PAGES = set(p for p, _ in ref_pages)  # 재실행한 82개 목록
print(f"kbcs 제외 페이지: {len(KBCS_REF_PAGES)}개")


# 형식: "파일명": 제외 시작 page (이 page부터 문서 끝까지 인덱스에서 제외)
CUT_FROM_PAGE = {
    "esmo_breast_cancer_patient_guide_korean.pdf": 60,
    "ncc_breast_cancer_screening_guideline_2015.pdf": 109,
    "nccn_metastatic_breast_cancer_patient.pdf": 63,
}

for fn, pg in CUT_FROM_PAGE.items():
    tot = total_pages.get(fn, "?")
    print(f"{fn}: p.{pg}~{tot} 제외 ({tot if tot=='?' else tot - pg}p 컷)")
if not CUT_FROM_PAGE:
    print("컷 페이지 미설정 — 위 스캔 결과를 보고 채우세요.")

kbcs 제외 페이지: 82개
esmo_breast_cancer_patient_guide_korean.pdf: p.60~70 제외 (10p 컷)
ncc_breast_cancer_screening_guideline_2015.pdf: p.109~114 제외 (5p 컷)
nccn_metastatic_breast_cancer_patient.pdf: p.63~66 제외 (3p 컷)


### 2번 최종 확정 요약

- **끝부분 컷 (CUT_FROM_PAGE)**: esmo p.60~ / ncc_screening p.109~ / nccn_metastatic p.63~ — 경계 육안 확인 완료 (p.59 본문 유지 확인).
- **kbcs**: 인용-밀도 필터(threshold 0.35)로 82개 페이지 제외 — 신규 걸린 페이지 샘플 육안 확인 결과 전부 인용 목록, 오탐 없음.
- 나머지 문서: 참고문헌 없음 → 전체 유지.

---
## 3. 경계 육안 확인 

컷 경계의 **직전 페이지(살아남는 마지막)와 컷 시작 페이지(지워지는 첫)**를 열어, 본문이 실수로 잘리지 않는지 확인한다.

In [9]:
def show_boundary(pdf_name, cut_page, ctx=1):
    pdf = next(p for p in pdf_files if p.name == pdf_name)
    docs = {d.metadata.get("page"): d for d in PyMuPDFLoader(str(pdf)).load()}
    print("=" * 70)
    print(f"### {pdf_name} — 컷 경계 p.{cut_page}")
    for pg in range(cut_page - ctx, cut_page + ctx + 1):
        if pg not in docs: continue
        tag = "[유지됨]" if pg < cut_page else "[제외됨]"
        text = docs[pg].page_content.strip()
        print(f"\n--- p.{pg} {tag} (앞 400자)")
        print(text[:400])
    print()

for fn, pg in CUT_FROM_PAGE.items():
    show_boundary(fn, pg)

### esmo_breast_cancer_patient_guide_korean.pdf — 컷 경계 p.60

--- p.59 [유지됨] (앞 400자)
60
유방암
지원 그룹
유방암 환자 옹호 그룹은 환자와 그 가족이 
유방암 환경을 파악할 수 있도록 돕습니다. 
그들은 지역 범위, 국가 범위 또는 국제적 
범위에 걸쳐 있을 수 있으며 환자가 적절하며 
시의적절한 치료와 교육을 받게 합니다. 이들 
그룹은 질병을 더 잘 이해하고 질병에 대처하는 
방법에 대해 배우고, 최상의 삶의 질을 유지하는 
데 필요한 도구를 최대한으로 제공할 수 
있습니다. 
•	
ABC Global Alliance: www.abcglobalalliance.org
•	
Advanced BC: http://advancedbc.org
•	
After Breast Cancer Diagnosis: www.abcdbreastcancersupport.org 
•	
Breast Cancer All

--- p.60 [제외됨] (앞 400자)
61
환자를 위한 ESMO 안내서
참고문헌
Balogun, O. D. and S. C. Formenti (2015). “Locally advanced breast cancer - strategies for developing nations.” 
Frontiers in oncology 5: 89.
Cancer.Net. (2016). “Fatigue.” Retrieved 12 Oct, 2017, from http://www.cancer.net/navigating-cancer-care/side-
effects/fatigue.
Cardoso, F., et al. (2018 [in press]). “Primary breast cancer: ESMO Clinical Practice Guidelines for diagn

--- p.61 [제외됨] (앞 400자)
62
유방암
Klastersky, J., et al. (2016). “Management of febrile ne

### 경계 확인 결과

- **esmo (컷 p.60)**: p.59는 본문(유방암 지원 그룹 안내)으로 온전히 유지, p.64는 파편 문자만 남은 부록/참고문헌 영역 — 경계 이상 없음.
- **kbcs**: 끝부분 컷 방식이 아니라 페이지 단위 필터(2-1)로 처리 — 걸린 페이지·갭 페이지 육안 확인 결과 전부 인용 목록, 본문 오탐 없음.
- **결론**: 컷 경계가 본문을 침범하지 않음을 확인. 4주차 클렌징(앵커 삭제) 실수 재발 없음.

---
## 4. 두 인덱스 구축 — 원본 vs 전처리 (컬렉션 week5pre_*)

In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

manifest_path = DATA_RAW / "metadata" / "manifest.json"
meta_lookup = {}
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        meta_lookup = {m["filename"]: m for m in json.load(f) if m.get("downloaded")}

def guess_lang(text):
    return "ko" if len(re.findall(r"[\uac00-\ud7a3]", text)) > 20 else "en"

def load_docs(apply_cut: bool):
    out = []
    for pdf in pdf_files:
        cut = CUT_FROM_PAGE.get(pdf.name) if apply_cut else None
        for d in PyMuPDFLoader(str(pdf)).load():
            pg = d.metadata.get("page", 0)
            if cut is not None and pg >= cut:
                continue  # 끝부분 컷 (esmo, ncc_screening, nccn_metastatic 등)
            if apply_cut and pdf.name == "kbcs_korean_breast_cancer_guideline_2023.pdf" and pg in KBCS_REF_PAGES:
                continue  # kbcs: 챕터별 산재 참고문헌 페이지 제외
            extra = meta_lookup.get(pdf.name, {})
            d.metadata.update({
                "filename": pdf.name, "org": extra.get("org", pdf.parent.name),
                "title": extra.get("title", pdf.stem),
                "language": extra.get("language", guess_lang(d.page_content)),
                "page": pg,
            })
            out.append(d)
    return out

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def split_docs(docs):
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = CHUNK_SIZE_BY_LANG.get(lang, 580); ov = CHUNK_OVERLAP_BY_LANG.get(lang, 85)
        sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=ov,
                                            separators=SEPARATORS, length_function=len)
        out.extend(sp.split_documents([d for d in docs if d.metadata.get("language") == lang]))
    return out

CONFIGS = {"P0_original": False, "P1_ref_removed": True}
chunk_store = {}
for name, cut in CONFIGS.items():
    ck = split_docs(load_docs(cut))
    chunk_store[name] = ck
    print(f"{name:16s} chunk수={len(ck)}")
print(f"\n제거된 chunk: {len(chunk_store['P0_original']) - len(chunk_store['P1_ref_removed'])}개")

P0_original      chunk수=2773
P1_ref_removed   chunk수=2240

제거된 chunk: 533개


In [11]:
import os as _os
from huggingface_hub import snapshot_download
_os.environ.pop("HF_HUB_OFFLINE", None); _os.environ.pop("TRANSFORMERS_OFFLINE", None)
_os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
except Exception:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(model_name=model_dir,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16})
print("embedding 로드 완료")

def build_retriever(name):
    vdir = VECTOR_ROOT / f"week5pre_{name}"; vdir.mkdir(parents=True, exist_ok=True)
    coll = f"breast_rag_week5pre_{name}"
    client = chromadb.PersistentClient(path=str(vdir))
    if coll in [c.name for c in client.list_collections()]:
        vs = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
        print(f"  {name}: 기존 재사용 ({vs._collection.count()}개)")
    else:
        ck = chunk_store[name]
        print(f"  {name}: 신규 인덱싱 {len(ck)}개 ...")
        vs = Chroma.from_documents(documents=ck, embedding=embeddings,
                                   collection_name=coll, persist_directory=str(vdir))
    return vs.as_retriever(search_kwargs={"k": TOP_K})

embedding 로드 완료


---
## 5. 생성 + RAGAS 비교 (judge=gpt-4o-mini) — 여기만 API 비용

In [12]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)
RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    return "\n\n---\n\n".join(
        f"[{i}] 출처: {d.metadata.get('org','?')} / {d.metadata.get('title','?')} / p.{d.metadata.get('page','?')}\n{d.page_content}"
        for i, d in enumerate(docs, 1))

golden = pd.read_csv(DATA_EVAL / "golden_set_v1.csv").to_dict("records")
print(f"golden_set: {len(golden)}문항")

golden_set: 30문항


In [13]:
import nest_asyncio; nest_asyncio.apply()
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]

def run_config(name):
    retriever = build_retriever(name)
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        docs = retriever.invoke(g["question"])
        ans = (llm | StrOutputParser()).invoke(
            RAG_PROMPT.format(context=format_context(docs), question=g["question"]))
        rows.append({"user_input": g["question"], "response": ans,
                     "retrieved_contexts": [d.page_content for d in docs],
                     "reference": g.get("ground_truth", "")})
    ds = EvaluationDataset.from_list(rows)
    df = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb).to_pandas()
    df.to_csv(DATA_PROCESSED / f"week5_ragas_{name}.csv", index=False, encoding="utf-8-sig")
    return df

score_tables = {}
for name in CONFIGS:
    score_tables[name] = run_config(name)
    print(f"{name}: 완료")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  P0_original: 신규 인덱싱 2773개 ...


RAG[P0_original]:   0%|          | 0/30 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
RAG[P0_original]: 100%|██████████| 30/30 [01:45<00:00,  3.52s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


P0_original: 완료
  P1_ref_removed: 신규 인덱싱 2240개 ...


RAG[P1_ref_removed]: 100%|██████████| 30/30 [01:38<00:00,  3.27s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

P1_ref_removed: 완료


---
## 6. 판정 — 제거 전 vs 후 (전체·언어별)

In [14]:
import re as _re
def _lang(q): return "KO" if _re.search("[가-힣]", str(q)) else "EN"

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]
rows = []
for name in CONFIGS:
    df = score_tables[name]
    row = {"config": name, "chunk수": len(chunk_store[name])}
    for c in METRIC_COLS:
        row[c] = round(df[c].mean(), 4)
    rows.append(row)
cmp = pd.DataFrame(rows)
for c in METRIC_COLS:
    cmp[c + "_delta"] = (cmp[c] - cmp[c].iloc[0]).round(4)
cmp.to_csv(DATA_PROCESSED / "week5_preprocess_comparison.csv", index=False, encoding="utf-8-sig")
print(cmp.to_string())

print("\n--- 언어별 context_precision ---")
for L in ["KO", "EN"]:
    for name in CONFIGS:
        df = score_tables[name].copy(); df["_l"] = df["user_input"].apply(_lang)
        sub = df[df["_l"] == L]
        print(f"  [{L}] {name:16s} CP={sub['context_precision'].mean():.4f}")
    print()

           config  chunk수  faithfulness  answer_relevancy  context_precision  faithfulness_delta  answer_relevancy_delta  context_precision_delta
0     P0_original    2773        0.7599            0.5510             0.8986              0.0000                  0.0000                    0.000
1  P1_ref_removed    2240        0.7775            0.6128             0.9006              0.0176                  0.0618                    0.002

--- 언어별 context_precision ---
  [KO] P0_original      CP=0.9571
  [KO] P1_ref_removed   CP=0.9602

  [EN] P0_original      CP=0.7815
  [EN] P1_ref_removed   CP=0.7815



## 판정 & 기록

- **한국어 CP 변화**: P0 0.9571 → P1 0.9602 (+0.003). 문항 단위로는 30문항 전체에서 **CP 하락 0개** (29 동일, 1 상승) — 참고문헌 제거가 어떤 문항의 검색도 해치지 않음.
- **판정**: **채택.** 인덱스 19% 감량(chunk 2,773 → 2,240)에도 세 지표 모두 개선 또는 동등 (faithfulness +0.018, answer_relevancy +0.062, CP +0.002). 4주차 클렌징(머리말 제거 → 악화)과 달리, 이번엔 검색에 기여하지 않는 '진짜 노이즈'만 제거됨을 실측으로 확인.
- **참고**: 문항별 faithfulness가 하락한 6문항은 모두 CP 불변(검색 동일) 상태에서의 변동이고, 상승 문항(8개)이 더 많아 제거 영향이 아닌 생성·채점의 회차 간 노이즈로 판단.
- **이후 처리**: P1 인덱스(`week5pre_P1_ref_removed`)를 5-1(Hybrid)부터의 baseline으로 사용. 특히 BM25는 참고문헌의 저자명·저널명 키워드에 오염되기 쉬우므로, Hybrid 도입 전 제거를 완료한 순서가 유효.